# Data, calibration and uncertainty {#sec-calibration}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c09-calibration.ipynb)

The hospital publishes one number a month: how many patients are waiting, by how long they have waited. It does not publish how many were referred or how many were treated. Yet every policy question is about those flows: how much capacity is there, how fast are referrals growing, and how much would more clinics help? This chapter estimates the flows and the model's parameters from the stock alone, measures how well the model fits, and puts honest uncertainty bands on what it says.

**Who it is for.** Anyone who fits a model to data or reads a fitted model's results. Data scientists will find familiar tools (least squares, train and test splits, the bootstrap) used on a dynamic model, where they behave differently. **Who can skip.** Nobody who will put numbers in front of a decision maker. Readers who know calibration well can go straight to sections 8 to 12, on identifiability and uncertainty. **Prerequisites.** Chapter 8 (Testing your model), and the ageing chain of chapter 4 (Delays, pipelines and memory). **Time.** About four hours with the exercises.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 9
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours (safe for colour-blind readers) with a distinct line style
# for each, so every figure also reads in greyscale
plt.rcParams["axes.prop_cycle"] = (
    cycler(color=["#000000", "#E69F00", "#56B4E9", "#009E73",
                  "#0072B2", "#D55E00", "#CC79A7"])
    + cycler(linestyle=["-", "--", ":", "-.", (0, (5, 1)),
                        (0, (3, 1, 1, 1)), (0, (1, 1))]))
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3})
print("loopbricks", lb.__version__, "| data:", len(wl), "months")

In [ ]:
from matplotlib.patches import FancyArrowPatch, Rectangle


def draw_case(ax):
    """The one-stock waiting list: what is observed and what is not."""
    pipe = dict(arrowstyle="-|>", mutation_scale=15, lw=3, color="0.3")
    hidden = dict(pipe, linestyle="--", color="0.55")
    ax.add_patch(Rectangle((0.4, 0.4), 0.2, 0.25, fill=False, lw=2.5))
    ax.text(0.5, 0.525, "Waiting list\n(observed monthly)", ha="center",
            va="center", fontsize=9)
    ax.add_patch(FancyArrowPatch((0.08, 0.525), (0.4, 0.525), **hidden))
    ax.add_patch(FancyArrowPatch((0.6, 0.525), (0.92, 0.525), **hidden))
    ax.text(0.22, 0.6, "Referrals\n(not published)", ha="center",
            fontsize=8)
    ax.text(0.78, 0.6, "Treatments\n(not published)", ha="center",
            fontsize=8)
    ax.text(0.78, 0.2, "MIN(Capacity, list / minimum treatment time)\n"
            "Capacity: trend x 2020 shock x 2022 initiative",
            ha="center", fontsize=8)
    ax.text(0.22, 0.25, "grows at a steady rate", ha="center", fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0.1, 0.8)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(8, 2.8))
draw_case(ax)
ax.set_title("Only the stock is observed")
plt.show()

Figure: the running case as a single stock. The list is observed every month; referrals and treatments, the two flows that change it, are not. Alt text: a box labelled waiting list between two dashed arrows, one flowing in and one flowing out.

**Learning outcomes.** By the end of the chapter you will be able to

1. infer what the stock data can and cannot say about the flows (Try its 1 and 2);
2. write a weighted payoff function and minimise it with a grid and a Nelder-Mead search you wrote yourself (Try it 5);
3. compute and interpret RMSE, MAE, MAPE, R² and the Theil inequality statistics, and avoid their traps (Try it 6);
4. diagnose a fit from its residuals (Try it 7);
5. detect parameters the data cannot identify, by profiling the payoff and reading correlations, and bring in evidence that separates them (Try it 9);
6. test a calibrated model out of sample, split in time (Try it 10);
7. estimate parameter uncertainty with a block bootstrap and turn it into prediction bands for a policy (Try it 12).

## 1. The data, and why they are synthetic

The series used throughout this book is **synthetic**: we made it with a short program, `notebooks/data/make_waiting_list.py`, and the program is part of the book. We did this for a practical reason and keep it for a teaching one.

The practical reason: the real series that inspired it, the National Treatment Purchase Fund's monthly outpatient counts for Ireland, is published on data.gov.ie, but the terms for reusing it in a book and its code are still being confirmed. Rather than wait, we built a series with the same shape and the same problem. It is monthly, it counts patients by months waited (0 to 3, 3 to 6, 6 to 12, and over 12), and it says nothing about additions to or removals from the list. When the licence is confirmed, the real series can be dropped in with the same columns.

The teaching reason: because we made the data, we know the truth. We can check whether a method recovers the parameters that produced the data, which is impossible with real data. Here is how the data were made, month by month, from January 2014 to December 2023:

* **Referrals** start at 4,000 patients a month, grow by 3% a year, and swing 4% above and below the trend with the seasons.
* **Capacity** starts at 3,950 patients a month and grows by 2% a year. In March 2020 it collapses to 35% of normal, then recovers in a straight line over nine months. From January 2022 a waiting-list initiative adds 10%.
* Both flows are multiplied each month by random noise with a standard deviation of about 3%.
* Patients **age** through the four bands: each month a third of the 0 to 3 and 3 to 6 bands, and a sixth of the 6 to 12 band, move on to the next band.
* **Treatments** are shared across the bands in proportion to their size, weighted 1.4, 1.0, 0.9 and 0.8 from the shortest wait to the longest (urgent cases are seen sooner).
* Counts are rounded to whole patients. The random seed is fixed, so the file is the same every time it is made.

In [ ]:
BAND_COLS = ["w0_3", "w3_6", "w6_12", "w12_plus"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(wl["month"], wl["total"] / 1000)
a1.set(title="Patients waiting", xlabel="month",
       ylabel="patients (thousands)")
for col in BAND_COLS:
    a2.plot(wl["month"], wl[col] / 1000, label=col)
a2.set(title="By months waited", xlabel="month",
       ylabel="patients (thousands)")
a2.legend()
plt.tight_layout()
plt.show()
print(wl.iloc[[0, 71, 72, 84, 95, 119]].to_string(index=False))

Figure: left, patients waiting, January 2014 to December 2023, in thousands. Right, the same by months waited. Alt text: a line that climbs slowly, jumps in 2020 and levels off after 2022; four band lines, of which the over-12-months band rises most.

The list grows from 32,276 patients in January 2014 to 43,839 in December 2019, jumps to 68,552 by December 2021 and ends 2023 at 69,927. Because we made the data, we can also write down the noise-free flows that produced them. This is the yardstick for everything that follows; with real data it would not exist.

In [ ]:
def true_flows(month):
    """The noise-free referrals and capacity used to make the data.

    month counts from 0 for January 2014 (patients/month).
    """
    month = np.asarray(month, dtype=float)
    calendar = (month % 12) + 1
    referrals = (4000 * 1.03 ** (month / 12)
                 * (1 + 0.04 * np.sin(2 * np.pi * (calendar - 3) / 12)))
    since = month - 74                               # March 2020 is month 74
    shock = np.where(since >= 0,
                     np.minimum(1, 0.35 + 0.65 * np.minimum(1, since / 9)),
                     1.0)
    capacity = (3950 * 1.02 ** (month / 12) * shock
                * np.where(month >= 96, 1.10, 1.0))
    return referrals, capacity


ref_true, cap_true = true_flows(np.arange(len(wl)))
print(f"2019 averages: referrals {ref_true[60:72].mean():,.0f}, capacity "
      f"{cap_true[60:72].mean():,.0f} patients/month")

In 2019 the true referrals averaged 4,700 patients a month and the true capacity 4,401.

**Try it 1.** *Check:* in which month is the list longest, and what share of it has waited over 12 months by December 2023? *Apply:* what was the true net flow (referrals minus capacity) in 2019? *Extend:* list two things you could do with this synthetic series that you could not do with the real one, and one thing you could do with the real one that you cannot do here.

<details><summary>Worked answer</summary>

*Check:* the longest list is the last, 69,927 in December 2023, and 50% of it has waited over 12 months. *Apply:* 299 patients a month. *Extend:* with the synthetic series you can check estimates against the truth and make as many new copies as you like with other seeds, to see how estimates vary. With the real series you can learn something about the real hospital: no synthetic series can surprise you with structure you did not put in.
</details>

In [ ]:
peak = wl.loc[wl["total"].idxmax()]
print(f"check: {len(wl)} months; highest list {peak.total:,} in "
      f"{peak.month:%b %Y}; over 12 months in Dec 2023: "
      f"{wl.w12_plus.iloc[-1] / wl.total.iloc[-1]:.0%}")
print(f"apply: true net flow in 2019: "
      f"{(ref_true - cap_true)[60:72].mean():,.0f} patients/month")

## 2. Inferring the flows the data do not show

A stock changes only through its flows (chapter 1), so the month-to-month **change in the list is the net flow**: referrals minus treatments. That much the data give us for free.

In [ ]:
net = wl["total"].diff()                       # patients/month
fig, ax = plt.subplots()
ax.plot(wl["month"], net, label="change in the list each month")
ax.plot(wl["month"], net.rolling(12, center=True).mean(),
        label="12-month average")
ax.plot(wl["month"], ref_true - cap_true, label="true net flow, no noise")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Net flow inferred from the stock",
       xlabel="month", ylabel="referrals minus treatments (patients/month)")
ax.legend()
plt.show()
pre = net[1:72]
print(f"2014 to 2019: mean {pre.mean():.0f}, standard deviation "
      f"{pre.std():.0f} patients/month")
when = wl.month[net.idxmax()]
print(f"largest monthly rise {net.max():,.0f} ({when:%b %Y})")

Figure: the change in the list each month (patients per month), its 12-month average, and the true noise-free net flow, January 2014 to December 2023. Alt text: a noisy line near zero that spikes in spring 2020 and hovers near zero after 2022.

From 2014 to 2019 the list grew by 163 patients a month on average, with a standard deviation of 217: the monthly noise is larger than the signal. The largest rise, 3,312 patients, came in March 2020.

Here is the catch. Any pair of referrals and treatments with the same difference gives the same list. 4,000 in and 3,900 out looks exactly like 6,000 in and 5,900 out. **From the total alone, only the difference is identified**, not the levels. Yet the levels matter: a 5% increase in capacity means 200 patients a month in one world and 300 in the other.

The bands carry more information, if we accept an assumption about ageing. Patients leave a band either by ageing into the next band or by being treated. In the oldest band nobody ages out, so whatever leaves it, beyond the arrivals from the 6 to 12 band, was treated. Taking the ageing rule of section 1 as known, the same bookkeeping gives the treatments from every band except the first.

In [ ]:
W = wl[BAND_COLS].to_numpy(dtype=float)
ageing = W[:-1] * np.array([1 / 3, 1 / 3, 1 / 6, 0])   # out of each band
arriving = np.c_[np.zeros(len(W) - 1), ageing[:, :3]]   # into the next
gone = W[:-1] + arriving - ageing - W[1:]       # treated (or referred in)
rate = gone[:, 1:] / W[:-1, 1:]                 # treated per patient
print("treated per patient per month, 2014 to 2019, bands 3_6, 6_12, 12+:",
      rate[:71].mean(axis=0).round(4))
print("relative to the 3 to 6 band:",
      (rate[:71].mean(axis=0) / rate[:71, 0].mean()).round(2))
print(f"band 0 to 3, referrals minus its treatments, 2019: "
      f"{-gone[59:71, 0].mean():,.0f} patients/month")

The inferred treatment rates for the three older bands are 0.116, 0.104 and 0.092 patients per patient waiting per month, in the ratio 1 : 0.90 : 0.80: exactly the weights used to make the data. The bookkeeping works.

The first band is different. Referrals enter it and treatments leave it, and the data show only the difference, 3,258 patients a month in 2019. To split it, we must assume how intensely the newest patients are treated compared with the 3 to 6 band, a **priority** factor. The table shows what each assumption implies.

In [ ]:
rows = []
for pi in ["1.0", "1.4", "2.0"]:
    treated_0 = float(pi) * rate[:, 0] * W[:-1, 0]
    referrals = -gone[:, 0] + treated_0
    rows.append({"priority assumed": pi,
                 "referrals, 2019 (patients/month)": referrals[59:71].mean(),
                 "treatments, 2019 (patients/month)":
                     (treated_0 + gone[:, 1:].sum(axis=1))[59:71].mean()})
rows.append({"priority assumed": "truth",
             "referrals, 2019 (patients/month)": ref_true[60:72].mean(),
             "treatments, 2019 (patients/month)": cap_true[60:72].mean()})
print(pd.DataFrame(rows).to_string(index=False, float_format="{:,.0f}".format))

With a priority of 1.4, the inferred 2019 referrals (4,692 a month) and treatments (4,434) are within 1% of the truth. Assume 1.0 or 2.0 instead and both flows shift by about 400 to 600 patients a month, while the list stays exactly the same. This is the **identifiability problem** in its plainest form. Data on the stock cannot fix the level of the flows without an assumption, and the assumption must come from somewhere else: a clinic audit, a referral count, or another data series. We return to it in sections 8 and 9.

**Try it 2.** *Check:* what was the average change in the list per month from 2014 to 2019? *Apply:* compare the inferred treatment rate of the over-12-months band in 2019, from April to December 2020, and in 2022. *Extend:* infer the 2022 referrals assuming a priority of 1.0 and of 2.0, and compare with the truth.

<details><summary>Worked answer</summary>

*Check:* 163 patients a month. *Apply:* 0.085 per month in 2019, 0.046 in April to December 2020 and 0.063 in 2022. The shock almost halved the chance that a long waiter was treated in a given month, and by 2022 it had not recovered, because the list had grown faster than capacity. *Extend:* 4,871 patients a month with a priority of 1.0 and 5,792 with 2.0, against a truth of 5,136. The range is wide, and only outside evidence on priority narrows it.
</details>

In [ ]:
print("apply: treated per patient per month in the 12-plus band:",
      f"2019 {rate[59:71, 2].mean():.3f}, Apr 2020 to Dec 2020 "
      f"{rate[74:83, 2].mean():.3f}, 2022 {rate[95:107, 2].mean():.3f}")
for pi in [1.0, 2.0]:
    refs = -gone[:, 0] + pi * rate[:, 0] * W[:-1, 0]
    print(f"extend: priority {pi}: referrals 2022 {refs[95:107].mean():,.0f}"
          " patients/month")
print(f"extend: truth 2022 {ref_true[96:108].mean():,.0f} patients/month")

## 3. A model to calibrate

**Calibration** means choosing the values of a model's parameters so that its output matches the data as closely as possible, in a sense we must define. We start with the running case as a single stock. Referrals grow at a steady rate; capacity grows, collapses in March 2020 and recovers, and gets a boost in January 2022; treatments are capacity unless the list is nearly empty. `TIME` counts months from the first count, at the end of January 2014, so the shock starts at month 73 and the initiative at month 95. An `Extra capacity` parameter, zero for now, lets us test a policy in section 12.

In [ ]:
def var(name, eq, units):
    return {"id": lb.name_key(name), "name": name, "eq": str(eq),
            "units": units}


CAPACITY = ("Capacity_0 * EXP(Capacity_growth * TIME / Months_per_year)"
            " * Shock_effect * (1 + STEP(Initiative_boost, Initiative_start))"
            " * (1 + STEP(Extra_capacity, Policy_start))")
SHOCK = ("1 - STEP(1, Shock_start) * (1 - Activity_at_worst)"
         " * MAX(0, 1 - (TIME - Shock_start) / Recovery_time)")
COMMON = [
    var("Capacity", CAPACITY, "patients/month"),
    var("Shock effect", SHOCK, "dmnl"),
    var("Referrals 0", 4000, "patients/month"),
    var("Referral growth", 0.03, "1/year"),
    var("Capacity 0", 3950, "patients/month"),
    var("Capacity growth", 0.02, "1/year"),
    var("Activity at worst", 0.5, "dmnl"),
    var("Recovery time", 6, "month"),
    var("Shock start", 73, "month"),
    var("Initiative boost", 0.05, "dmnl"),
    var("Initiative start", 95, "month"),
    var("Extra capacity", 0, "dmnl"),
    var("Policy start", 119, "month"),
    var("Minimum treatment time", 0.5, "month"),
    var("Months per year", 12, "month/year"),
]
REFERRALS = "Referrals_0 * EXP(Referral_growth * TIME / Months_per_year)"
TREATMENTS = "MIN(Capacity, Waiting_list / Minimum_treatment_time)"
one_stock = lb.Model.from_spec({
    "name": "Waiting list", "timeUnit": "month", "T": 119, "dt": 0.25,
    "stocks": [{"id": "wl", "name": "Waiting list",
                "init": str(wl.total[0]), "units": "patients",
                "nonNegative": True}],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl", "eq": REFERRALS,
         "units": "patients/month"},
        {"id": "tr", "name": "Treatments", "from": "wl", "eq": TREATMENTS,
         "units": "patients/month"}],
    "vars": COMMON})
print(one_stock.equations())

Seven parameters are to be estimated: `referrals_0`, `referral_growth`, `capacity_0`, `capacity_growth`, `activity_at_worst` (capacity as a fraction of normal at the bottom of the shock), `recovery_time` and `initiative_boost`. Each has a plausible range, set before looking at the fit, as in chapter 8. The first guess below uses round numbers.

In [ ]:
PARAMS = ["referrals_0", "referral_growth", "capacity_0", "capacity_growth",
          "activity_at_worst", "recovery_time", "initiative_boost"]
LOW = np.array([3000, 0.0, 3000, -0.02, 0.2, 3, 0.0])
HIGH = np.array([5000, 0.05, 5000, 0.05, 0.6, 18, 0.2])
guess = np.array([one_stock.params[p] for p in PARAMS])
obs = wl["total"].to_numpy(dtype=float)
MONTHS = np.arange(len(wl))


def simulate(p, T=119, model=one_stock, names=PARAMS):
    """Monthly list (patients) for parameter vector p, TIME 0 to T."""
    run = model.run(T=T, include=["Waiting list"], warn_clips=False,
                    **dict(zip(names, p)))
    return run["Waiting list"].to_numpy()[::4]      # dt = 0.25: every 4th


first = simulate(guess)
fig, ax = plt.subplots()
ax.plot(wl["month"], obs / 1000, label="data")
ax.plot(wl["month"], first / 1000, label="first guess")
ax.set(title="Before calibration", xlabel="month",
       ylabel="patients waiting (thousands)")
ax.legend()
plt.show()
print(f"first guess: worst month {np.abs(first - obs).max():,.0f} patients "
      "off")

Figure: the data and the model with first-guess parameters, January 2014 to December 2023, in thousands of patients. Alt text: two rising lines; the model's line falls short of the data after 2020.

The first guess captures the shape but misses by up to 7,319 patients. Calibration will do better; the question is how to say what "better" means.

## 4. The payoff function

A **payoff function** turns a run into one number that the search tries to make as small as possible. The standard choice is the **weighted sum of squared errors**:

$$P(\theta) = \sum_{t} w_t \left(\frac{\hat{y}_t(\theta) - y_t}{\sigma}\right)^2$$

where $\theta$ is the vector of parameters, $\hat{y}_t(\theta)$ is the simulated list in month $t$, $y_t$ is the data, $\sigma$ is the size of error we regard as ordinary, and $w_t$ is a weight for each month (1 unless a month should count more or less).

Three choices are hidden in that formula, and each should be made on purpose [@oliva2003].

* **Squared errors** punish large misses heavily, which suits data whose errors are roughly normal. Absolute errors are less sensitive to outliers.
* **The error scale** $\sigma$. With one series it does not move the best fit, but it makes the payoff readable: if the model misses by about $\sigma$ each month, the payoff is about the number of months. With several series (section 9) it sets their relative weight. We use the standard deviation of the monthly changes in the list, 594 patients.
* **The weights** $w_t$ let some months count more: recent months, if the model is for forecasting, or the months around a policy change, if that is what it must explain.

Finally, the payoff returns infinity outside the plausible ranges, so the search cannot wander into values that make no physical sense.

In [ ]:
SIGMA = float(np.diff(obs).std())       # patients: the error scale we accept
print(f"sigma = {SIGMA:.0f} patients")


def payoff(p, months=MONTHS, data=obs, weights=None):
    """Weighted sum of squared errors over the given months.

    Each error is divided by SIGMA; a weight per month (default 1) lets
    some months count more. Outside the plausible ranges: infinity.
    """
    if np.any(p < LOW) or np.any(p > HIGH):
        return np.inf
    sim = simulate(p, T=int(months[-1]))
    w = 1.0 if weights is None else weights[months]
    return float((w * ((sim[months] - data[months]) / SIGMA) ** 2).sum())


print(f"payoff of the first guess: {payoff(guess):,.0f}")

## 5. Searching: a grid, then Nelder-Mead

With two parameters, the simplest search is a **grid**: try every combination and keep the best. `run_batch` makes this cheap, running all 1,681 combinations of `referrals_0` and `capacity_0` together, with the other parameters at their first guesses.

In [ ]:
grid_r = np.linspace(3600, 4800, 41)
grid_c = np.linspace(3600, 4800, 41)
R, C = np.meshgrid(grid_r, grid_c)
settings = pd.DataFrame({"referrals_0": R.ravel(), "capacity_0": C.ravel()})
out = one_stock.run_batch(settings, include=["Waiting list"],
                          warn_clips=False)
sims = out.pivot(index="time", columns="run", values="value").to_numpy()
sse = (((sims[::4] - obs[:, None]) / SIGMA) ** 2).sum(axis=0)
surface = sse.reshape(R.shape)
k = int(np.argmin(sse))
print(f"best on the grid: referrals_0 {R.ravel()[k]:.0f}, capacity_0 "
      f"{C.ravel()[k]:.0f}, payoff {sse[k]:,.0f}")
fig, ax = plt.subplots(figsize=(6, 5))
cs = ax.contour(R, C, np.log10(surface), levels=12, colors="black",
                linewidths=0.8)
ax.clabel(cs, fmt="%.1f", fontsize=7)
ax.plot(R.ravel()[k], C.ravel()[k], "o", color="#D55E00")
ax.set(title="Payoff (log10) over two parameters",
       xlabel="referrals_0 (patients/month)",
       ylabel="capacity_0 (patients/month)")
plt.show()

Figure: contours of the payoff (log scale) over `referrals_0` and `capacity_0`, both in patients per month, with the best grid point marked. Alt text: contour lines that run in long parallel bands along the diagonal.

The contours run along the diagonal: raising referrals and capacity together hardly changes the payoff, exactly as section 2 predicted. The best grid point, 4,590 and 4,530 patients a month with a payoff of 1,315, is just one point in a long valley.

A grid is hopeless with seven parameters: ten values each would need ten million runs. The **Nelder-Mead** method [@nelder1965] needs no derivatives and only a few hundred runs. It keeps a **simplex** of eight points (one more than the number of parameters) and repeatedly replaces the worst point. It first tries the worst point's mirror image through the centre of the others (reflection). If that is very good, it goes further (expansion); if poor, it tries a point part way back (contraction); and if nothing helps, it shrinks the whole simplex towards the best point. Thirty lines of numpy are enough.

In [ ]:
def nelder_mead(f, x0, step, max_evals=3000, tol=1e-9):
    """Minimise f from x0 with the Nelder-Mead simplex method.

    step: the size of the first simplex along each axis.
    Returns the best point, its value and the number of evaluations.
    """
    n = len(x0)
    X = np.array([x0] + [x0 + np.eye(n)[i] * step[i] for i in range(n)],
                 dtype=float)
    F = np.array([f(x) for x in X])
    evals = n + 1
    while evals < max_evals:
        order = np.argsort(F)
        X, F = X[order], F[order]                 # best first, worst last
        if F[-1] - F[0] <= tol * (abs(F[0]) + 1e-12):
            break
        centre = X[:-1].mean(axis=0)              # of all but the worst
        xr = centre + (centre - X[-1])            # reflect the worst
        fr = f(xr)
        evals += 1
        if fr < F[0]:                             # very good: go further
            xe = centre + 2 * (centre - X[-1])
            fe = f(xe)
            evals += 1
            X[-1], F[-1] = (xe, fe) if fe < fr else (xr, fr)
        elif fr < F[-2]:                          # good enough: keep it
            X[-1], F[-1] = xr, fr
        else:                                     # poor: pull back in
            if fr < F[-1]:
                xc = centre + 0.5 * (xr - centre)
            else:
                xc = centre + 0.5 * (X[-1] - centre)
            fc = f(xc)
            evals += 1
            if fc < min(fr, F[-1]):
                X[-1], F[-1] = xc, fc
            else:                                 # shrink towards the best
                X[1:] = X[0] + 0.5 * (X[1:] - X[0])
                F[1:] = [f(x) for x in X[1:]]
                evals += n
    i = int(np.argmin(F))
    return X[i], F[i], evals

We start from the best grid point and run the search twice, the second time with a smaller simplex, a cheap guard against stopping early.

In [ ]:
start = guess.copy()
start[[0, 2]] = R.ravel()[k], C.ravel()[k]          # from the grid
t0 = time.time()
fit, f_fit, n_evals = nelder_mead(payoff, start, 0.1 * (HIGH - LOW))
fit, f_fit, more = nelder_mead(payoff, fit, 0.02 * (HIGH - LOW))
print(f"{n_evals + more} runs in {time.time() - t0:.1f} s; payoff "
      f"{f_fit:.1f} (first guess {payoff(guess):,.0f})")
fitted = simulate(fit)
print(pd.DataFrame({"estimate": fit, "low": LOW, "high": HIGH},
                   index=PARAMS).to_string(float_format="{:,.4g}".format))
fig, ax = plt.subplots()
ax.plot(wl["month"], obs / 1000, label="data")
ax.plot(wl["month"], fitted / 1000, label="calibrated model")
ax.set(title="After calibration to the total", xlabel="month",
       ylabel="patients waiting (thousands)")
ax.legend()
plt.show()

Figure: the data and the calibrated model, January 2014 to December 2023, in thousands of patients. Alt text: two lines that lie almost on top of each other.

After 2,018 runs the payoff has fallen from 3,989 to 70.9, and the calibrated line lies almost on the data. But look at the estimates before celebrating. `referral_growth` has run to the top of its plausible range (5% a year), and `capacity_growth` (4.3%) is not far behind. When an estimate sits on a bound, the data are pushing it somewhere implausible, and that is a warning, not a result. Section 8 explains what is going on.

**Try it 5.** *Check:* by how much did calibration reduce the payoff? *Apply:* give the months from 2020 onwards three times the weight and refit, starting from the fit. Do the estimates change? *Extend:* start Nelder-Mead from three random points in the plausible box (seeds 0, 1 and 2) and compare the `capacity_0` they find and their payoffs.

<details><summary>Worked answer</summary>

*Check:* from 3,989 to 70.9. *Apply:* hardly at all (`capacity_0` moves from 4,781 to 4,791 patients a month); the fit was already good in those months. *Extend:* the three searches find `capacity_0` of 3,255, 3,276 and 3,452 patients a month, with payoffs of 77.6, 71.3 and 71.2. Two of them fit almost as well as our best (70.9) with a capacity 1,300 patients a month lower. A good fit does not mean a well-determined parameter.
</details>

In [ ]:
recent = np.where(MONTHS >= 72, 3.0, 1.0)
f_recent = lambda p: payoff(p, weights=recent)
fit_r, _, _ = nelder_mead(f_recent, fit, 0.02 * (HIGH - LOW))
print("apply:", pd.DataFrame({"equal weights": fit, "2020 on x 3": fit_r},
                             index=PARAMS).to_string(
                                 float_format="{:,.4g}".format))
for seed in range(3):
    r = np.random.default_rng(seed)
    x0 = LOW + r.uniform(0.3, 0.7, len(LOW)) * (HIGH - LOW)
    x, fx, _ = nelder_mead(payoff, x0, 0.1 * (HIGH - LOW))
    x, fx, _ = nelder_mead(payoff, x, 0.02 * (HIGH - LOW))
    print(f"extend: start {seed}: capacity_0 {x[2]:,.0f}, payoff {fx:.1f}")

## 6. How good is the fit: performance measures

A single payoff is not enough to judge a fit. Here are the measures in common use, what each tells you, and where each misleads. Write $e_t = \hat{y}_t - y_t$ for the error in month $t$ and $n$ for the number of months.

* **RMSE**, root mean squared error, $\sqrt{\tfrac{1}{n}\sum e_t^2}$, in the units of the data (patients). It weights large errors heavily.
* **MAE**, mean absolute error, $\tfrac{1}{n}\sum |e_t|$, also in patients, and less driven by a few bad months.
* **MAPE**, mean absolute percentage error, $\tfrac{100}{n}\sum |e_t / y_t|$. Unit-free, so it compares series of different sizes, but it has traps (below).
* **R²**, $1 - \sum e_t^2 / \sum (y_t - \bar{y})^2$: the share of the variation around the mean that the model explains.
* **Theil inequality statistics** split the mean squared error into three parts that tell the modeller *what kind* of error it is [@sterman2000]:

$$\frac{1}{n}\sum e_t^2 = (\bar{S} - \bar{A})^2 + (s_S - s_A)^2 + 2(1 - r)\, s_S\, s_A$$

where $\bar{S}, \bar{A}$ are the means of the simulated and actual series, $s_S, s_A$ their standard deviations and $r$ their correlation. Dividing each term by the mean squared error gives three fractions that add up to 1:

* $U^M$, **bias**: the model is systematically too high or too low. A large $U^M$ usually means a parameter is wrong, and it is worth fixing.
* $U^S$, **unequal variance**: the model's series varies more or less than the data, for example a trend that is too steep or swings that are too small. Also systematic, and also worth fixing.
* $U^C$, **unequal covariation**: the point-by-point errors left when the means and spreads match. If most of the error is here, the model captures the systematic behaviour and misses only noise, or a short shift in timing. A model of the main dynamics need not, and should not, chase that.

We compare the calibrated model with two benchmarks that have no feedback structure at all: a straight line fitted to the whole series, and the 2014 to 2019 average.

In [ ]:
def measures(sim, data):
    """Goodness-of-fit measures of a simulated series against data."""
    sim, data = np.asarray(sim, float), np.asarray(data, float)
    err = sim - data
    mse = np.mean(err ** 2)
    s_sd, d_sd = sim.std(), data.std()
    r = np.corrcoef(sim, data)[0, 1]
    return {
        "RMSE": np.sqrt(mse),
        "MAE": np.mean(np.abs(err)),
        "MAPE (%)": 100 * np.mean(np.abs(err / data)),
        "R2": 1 - np.sum(err ** 2) / np.sum((data - data.mean()) ** 2),
        "U_M (bias)": (sim.mean() - data.mean()) ** 2 / mse,
        "U_S (variance)": (s_sd - d_sd) ** 2 / mse,
        "U_C (covariance)": 2 * (1 - r) * s_sd * d_sd / mse,
    }


line = np.polyval(np.polyfit(MONTHS, obs, 1), MONTHS)
flat = np.full_like(obs, obs[:72].mean())
table = pd.DataFrame({"calibrated model": measures(fitted, obs),
                      "straight line": measures(line, obs),
                      "2014 to 2019 average": measures(flat, obs)})
print(table.to_string(float_format="{:,.3f}".format))

The calibrated model misses by 456 patients (RMSE) or 375 (MAE) in a typical month, 0.87% of the list, and almost all of its error (98%) is unequal covariation: noise, not structure.

The benchmarks show why no single measure is enough.

* **R² flatters trends.** A straight line scores an R² of 0.880 on a series that it misses by 5,113 patients a month, because the trend alone explains most of the variation. Any trending series gives a high R². Judge R² against a benchmark, never on its own.
* **Theil tells you what to fix.** The line has no bias ($U^M = 0$: least squares makes its mean errors cancel) and 97% of its error in $U^C$. That looks like "only noise", but the noise is the 2020 jump the line cannot follow. So read Theil fractions together with the size of the error: a large RMSE with a large $U^C$ means a pattern is missing. The 2014 to 2019 average is all bias and unequal variance, the signature of a model that misses the trend entirely.

MAPE has traps of its own, which show up on a series that crosses zero. The net flow is one.

In [ ]:
net_model = np.diff(fitted)
net_data = np.diff(obs)
print("net flow (patients/month), calibrated model:")
print(pd.Series(measures(net_model, net_data)).round(2).to_string())
small = np.abs(net_data) < 50
ape = 100 * np.abs((net_model - net_data) / net_data)
print(f"months with a net flow under 50 patients: {small.sum()}; their "
      f"mean absolute percentage error: {ape[small].mean():,.0f}%")
for forecast in [0, 50, 150, 300]:
    print(f"actual 100, forecast {forecast:3}: "
          f"APE {100 * abs(forecast - 100) / 100:.0f}%")

On the net flow the same model has an R² of 0.88 but a MAPE of 293%. In the 15 months when the net flow was under 50 patients, a small error in patients becomes a huge percentage: 1,832% on average. MAPE is **undefined at zero and explodes near it**. It is also **asymmetric**: a forecast that is too low can never score worse than 100%, but one that is too high has no limit. A search that minimises MAPE therefore favours forecasts that are too low. Use MAPE only for series well away from zero, and report it alongside a measure in the data's own units.

**Try it 6.** *Check:* compute the measures for the first guess of section 3. What kind of error dominates? *Apply:* the symmetric MAPE divides each error by the average of the forecast and the actual. Compute it for the calibrated model. *Extend:* compute the measures for 2014 to 2019 only. Why is R² lower there, when the RMSE is about the same?

<details><summary>Worked answer</summary>

*Check:* RMSE 3,423 patients, MAPE 4.01% and R² 0.946, with 23% of the error in bias, 54% in unequal variance and only 23% in covariation. The first guess is systematically wrong: its rise after 2020 is too small. Systematic error is the kind calibration should remove, and it did. *Apply:* 0.873%, against 0.872% for MAPE. Far from zero the two agree; they differ only where MAPE misbehaves. *Extend:* R² is 0.981 against 0.999, with an RMSE of 472 patients. Before 2020 the list varies much less, so the same error is a larger share of the variation. R² depends on the data's spread as much as on the model.
</details>

In [ ]:
print("check: the first guess:")
print(pd.Series(measures(first, obs)).round(3).to_string())
smape = 100 * np.mean(2 * np.abs(fitted - obs)
                      / (np.abs(fitted) + np.abs(obs)))
print(f"apply: MAPE {table.loc['MAPE (%)', 'calibrated model']:.3f}%, "
      f"symmetric MAPE {smape:.3f}%")
early = measures(fitted[:72], obs[:72])
print("extend: 2014 to 2019 only:")
print(pd.Series(early).round(3).to_string())

## 7. Reading the residuals

The **residuals**, data minus model, should look like noise if the model has captured the systematic behaviour. Two checks catch most problems. First, plot them against time, looking for trends, steps or seasons. Second, compute their **autocorrelation**: the correlation of the residuals with themselves a given number of months earlier. For independent noise, about 95% of the autocorrelations should lie within $\pm 2/\sqrt{n}$. The **Durbin-Watson statistic**, $\sum (e_t - e_{t-1})^2 / \sum e_t^2$, condenses lag 1 into one number: about 2 for independent errors, near 0 for strongly positive autocorrelation.

In [ ]:
res = obs - fitted


def acf(x, lags=24):
    """Autocorrelation of x at lags 1 to `lags`."""
    x = x - x.mean()
    return np.array([np.sum(x[k:] * x[:-k]) / np.sum(x * x)
                     for k in range(1, lags + 1)])


rho = acf(res)
band = 2 / np.sqrt(len(res))
dw = np.sum(np.diff(res) ** 2) / np.sum(res ** 2)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(wl["month"], res)
a1.axhline(0, color="black", linewidth=0.8)
a1.set(title="Residuals (data minus model)", xlabel="month",
       ylabel="patients")
a2.bar(range(1, 25), rho, color="#56B4E9", edgecolor="black")
a2.axhline(band, color="black", linestyle="--")
a2.axhline(-band, color="black", linestyle="--")
a2.set(title="Autocorrelation of the residuals", xlabel="lag (months)",
       ylabel="autocorrelation")
plt.tight_layout()
plt.show()
print(f"lag 1: {rho[0]:.2f}, lag 12: {rho[11]:.2f}, bounds +/- {band:.2f};"
      f" Durbin-Watson {dw:.2f}")
print(f"runs of same-sign residuals: "
      f"{1 + int(np.sum(np.sign(res[1:]) != np.sign(res[:-1])))} in "
      f"{len(res)} months")

Figure: left, the residuals of the calibrated model over time, in patients. Right, their autocorrelation for lags of 1 to 24 months, with dashed bounds at $\pm 0.18$. Alt text: a slowly wandering residual line; bars that start near 0.9 and decline over several months.

The residuals wander in long swings: there are only 14 runs of same-sign residuals in 120 months, the lag-1 autocorrelation is 0.89, and the Durbin-Watson statistic is 0.21. These are not independent errors. The reason is in how the data were made: the noise hits the *flows*, and a stock adds up its flows, so each month's error is carried into every later month. A deterministic model cannot follow that random walk, and should not try.

This matters for what comes next. Methods that assume independent errors, such as the usual formulae for standard errors, will be far too confident. The lag-12 autocorrelation is only 0.02, so the missing seasonality in referrals is too small to see in the levels, though Try it 7 finds it in the changes.

**Try it 7.** *Check:* at what lag does the autocorrelation first fall inside the bounds, and how many residuals are positive? *Apply:* compute the autocorrelation of the *changes* in the residuals at lags 1, 6 and 12. *Extend:* with a lag-1 autocorrelation $\rho$, $n$ correlated months carry about as much information as $n(1-\rho)/(1+\rho)$ independent ones. How many is that here?

<details><summary>Worked answer</summary>

*Check:* at lag 5; 64 of the 120 residuals are positive, so there is no overall bias. *Apply:* 0.22 at lag 1, minus 0.24 at lag 6 and 0.20 at lag 12, all just outside the bounds of $\pm 0.18$. A positive value at 12 months and a negative one at 6 is the signature of a season: the model leaves out the 4% seasonal swing in referrals. *Extend:* about 7. Ten years of monthly data carry the information of only about seven independent observations about the level of the list, which is why parameter uncertainty in section 11 needs a method that respects the correlation.
</details>

In [ ]:
first_out = int(np.argmax(np.abs(rho) < band)) + 1
print(f"check: first lag inside the bounds: {first_out} months; "
      f"positive residuals: {(res > 0).sum()} of {len(res)}")
rho_net = acf(np.diff(res))
print(f"apply: residual net flow, lag 1 {rho_net[0]:.2f}, lag 6 "
      f"{rho_net[5]:.2f}, lag 12 {rho_net[11]:.2f} (bounds +/- "
      f"{2 / np.sqrt(len(res) - 1):.2f})")
n_eff = len(res) * (1 - rho[0]) / (1 + rho[0])
print(f"extend: effective number of independent months about {n_eff:.0f}")

## 8. Did we find the truth? Identifiability

Now the advantage of synthetic data pays off. We can compare the estimates with the values that made the data. (In this model's time, the true starting values are a month's growth above the generator's 4,000 and 3,950, and the true growth rates are $\ln 1.03$ and $\ln 1.02$ per year.)

A first tool for diagnosis is the **Jacobian**: how much every simulated month moves when each parameter is nudged. One `run_batch` call with eight runs gives it. If the errors were independent, the standard errors and correlations of the estimates would follow from it.

In [ ]:
TRUTH = np.array([4000 * 1.03 ** (1 / 12), np.log(1.03),
                  3950 * 1.02 ** (1 / 12), np.log(1.02), 0.35, 9, 0.10])


def jacobian(p, model=one_stock, names=PARAMS, T=119):
    """Sensitivity of every monthly value to each parameter (run_batch)."""
    h = 1e-4 * np.maximum(np.abs(p), 1e-2)
    rows = [p] + [p + np.eye(len(p))[i] * h[i] for i in range(len(p))]
    out = model.run_batch(pd.DataFrame(rows, columns=names), T=T,
                          include=["Waiting list"], warn_clips=False)
    s = out.pivot(index="time", columns="run", values="value").to_numpy()
    s = s[::4]
    return (s[:, 1:] - s[:, [0]]) / h


def standard_errors(J, residuals, n_params):
    """Standard errors and correlations from J, assuming independent errors."""
    s2 = np.sum(residuals ** 2) / (len(residuals) - n_params)
    cov = s2 * np.linalg.inv(J.T @ J)
    se = np.sqrt(np.diag(cov))
    return se, cov / np.outer(se, se)


J = jacobian(fit)
se, corr = standard_errors(J, res, len(fit))
print(pd.DataFrame({"truth": TRUTH, "estimate": fit, "std error": se},
                   index=PARAMS).to_string(float_format="{:,.4g}".format))
print("correlation of the estimates:")
print(pd.DataFrame(corr, index=PARAMS, columns=PARAMS).round(2).to_string())

The estimates are far from the truth: capacity of 4,781 patients a month against 3,957, growth rates well above the true ones, and activity at worst 0.54 against 0.35. And the model knows it cannot tell: the standard error on `capacity_0` is about 125,000 patients a month, and `referrals_0`, `capacity_0`, `activity_at_worst` and `initiative_boost` are correlated at plus or minus 1.00. A correlation of 1 means the data cannot separate the parameters: changing one can be undone exactly by changing the others.

The cleanest picture is a **profile** of the payoff. Hold one parameter fixed at a series of values, refit all the others each time, and plot the best payoff that remains. A sharp valley means the parameter is identified; a flat line means it is not. Because the grid showed a diagonal valley, each refit starts with `referrals_0` moved by the same amount as `capacity_0`.

In [ ]:
def profile(values, payoff_fn, fit, index, step, start_for, evals=1500):
    """Best payoff with parameter `index` held at each value in turn.

    start_for(v) gives a starting point for the other parameters.
    """
    best, points = [], []
    for v in values:
        f = lambda z: payoff_fn(np.insert(z, index, v))
        x0 = np.delete(start_for(v), index)
        x, fx, _ = nelder_mead(f, x0, np.delete(step, index), evals)
        x, fx, _ = nelder_mead(f, x, np.delete(step, index) / 4, evals)
        best.append(fx)
        points.append(np.insert(x, index, v))
    return np.array(best), np.array(points)


def along_ridge(v, fit=fit):
    """Start with referrals_0 moved as far as capacity_0: same gap."""
    x = fit.copy()
    x[0] += v - fit[2]
    x[2] = v
    return x


cap_values = np.array([3600, 3800, 4000, 4200, 4400, 4600, 4800])
t0 = time.time()
prof_total, ridge = profile(cap_values, payoff, fit, 2, 0.05 * (HIGH - LOW),
                     along_ridge)
print(f"{time.time() - t0:.0f} s")
print(pd.Series(prof_total - f_fit, index=cap_values,
                name="payoff above the best").round(2).to_string())

Across the whole plausible range, from 3,600 to 4,800 patients a month, the best payoff changes by less than 0.3. Every one of those capacities fits the total equally well. The parameter is **not identifiable** from the total, and the "best fit" in section 5 was just where the search happened to stop.

The reason is the one found in section 2, with a twist. Before 2020 only the gap between referrals and capacity shows in the data. In 2020 the drop in capacity reveals $(1 - \text{activity at worst}) \times \text{capacity}$, not the two separately: a large capacity with a mild shock looks the same as a small capacity with a deep one. The 2022 initiative likewise reveals only boost times capacity.

## 9. The bands as extra evidence

The four bands add information the total cannot carry. When treatments collapse, patients pile up in the older bands at a rate that depends on how many were being treated per patient waiting, and that rate is tied to the *level* of capacity, not just its changes. So we calibrate the model of chapter 8, with the list in four bands, to all five series at once: the total and each band. Each series gets its own error scale $\sigma_k$ (the standard deviation of its monthly changes), so that a band with small, steady changes is not swamped by the total:

$$P(\theta) = \sum_{k} \sum_{t} \left(\frac{\hat{y}_{k,t}(\theta) - y_{k,t}}{\sigma_k}\right)^2$$

One more parameter enters: `priority`, the weight on the newest band.

In [ ]:
BANDS = ["Waiting 0 to 3", "Waiting 3 to 6", "Waiting 6 to 12",
         "Waiting 12 plus"]
band_stocks = [{"id": f"b{i}", "name": b, "init": str(wl[c][0]),
                "units": "patients", "nonNegative": True}
               for i, (b, c) in enumerate(zip(BANDS, BAND_COLS))]
keys = [lb.name_key(b).capitalize() for b in BANDS]
band_flows = [{"id": "ref", "name": "Referrals", "to": "b0",
               "eq": REFERRALS, "units": "patients/month"}]
for i in range(3):
    band_flows.append({"id": f"a{i}", "name": f"Ageing {i}",
                       "from": f"b{i}", "to": f"b{i + 1}",
                       "eq": f"{keys[i]} / Band_time_{i}",
                       "units": "patients/month"})
for i in range(4):
    weight = "Priority * " if i == 0 else ""
    band_flows.append({"id": f"t{i}", "name": f"Treating {i}",
                       "from": f"b{i}",
                       "eq": f"Treatments * {weight}{keys[i]} / Weighted_list",
                       "units": "patients/month"})
band_vars = COMMON + [
    var("Waiting list", " + ".join(keys), "patients"),
    var("Weighted list",
        f"MAX(Priority * {' + '.join(keys)}, Smallest_list)", "patients"),
    var("Treatments", TREATMENTS, "patients/month"),
    var("Priority", 1.4, "dmnl"),
    var("Band time 0", 3, "month"), var("Band time 1", 3, "month"),
    var("Band time 2", 6, "month"), var("Smallest list", 1, "patients"),
]
band_model = lb.Model.from_spec({
    "name": "Waiting list by band", "timeUnit": "month", "T": 119,
    "dt": 0.25, "stocks": band_stocks, "flows": band_flows,
    "vars": band_vars})
print(band_model.equations().split("Variables:")[0])

In [ ]:
PB = PARAMS + ["priority"]
LOW_B, HIGH_B = np.r_[LOW, 1.0], np.r_[HIGH, 3.0]
OBS = wl[["total"] + BAND_COLS].to_numpy(dtype=float)     # months x 5
SIG = np.diff(OBS, axis=0).std(axis=0)                    # one per series
print("sigma (patients):", pd.Series(SIG, index=["total"] + BAND_COLS)
      .round().to_dict())


def simulate_bands(rows, T=119):
    """Total and four bands, months x 5 x runs, for parameter rows."""
    out = band_model.run_batch(pd.DataFrame(np.atleast_2d(rows), columns=PB),
                               T=T, include=BANDS, warn_clips=False)
    arr = np.stack([out[out.variable == b].pivot(
        index="time", columns="run", values="value").to_numpy()[::4]
        for b in BANDS], axis=1)
    return np.concatenate([arr.sum(axis=1, keepdims=True), arr], axis=1)


def simulate_one(p, T=119):
    """Total and four bands, months x 5, for one parameter vector."""
    run = band_model.run(T=T, include=BANDS, warn_clips=False,
                         **dict(zip(PB, p)))
    arr = run.to_numpy()[::4]
    return np.c_[arr.sum(axis=1), arr]


def payoff_bands(p, months=MONTHS, data=OBS, use=slice(None)):
    """Weighted SSE over the total and the bands (use picks the series)."""
    if np.any(p < LOW_B) or np.any(p > HIGH_B):
        return np.inf
    sim = simulate_one(p, T=int(months[-1]))
    z = (sim[months] - data[months]) / SIG
    return float((z[:, use] ** 2).sum())


t0 = time.time()
start_b = np.r_[guess, 1.4]
fit_b, f_b, n1 = nelder_mead(payoff_bands, start_b, 0.1 * (HIGH_B - LOW_B))
fit_b, f_b, n2 = nelder_mead(payoff_bands, fit_b, 0.02 * (HIGH_B - LOW_B))
print(f"{n1 + n2} runs in {time.time() - t0:.0f} s; payoff {f_b:,.1f}")
fitted_b = simulate_one(fit_b)
rmse_b = np.sqrt(((fitted_b - OBS) ** 2).mean(axis=0))
print("RMSE by series (patients):",
      pd.Series(rmse_b, index=["total"] + BAND_COLS).round().to_dict())

The fit to the total is a little worse than before (RMSE 499 patients against 456), because the model must now also match the bands, with RMSEs from 146 patients (3 to 6 months) to 407 (0 to 3 months).

In [ ]:
def jacobian_bands(p):
    """d(simulated / SIG)/d(parameter), stacked over months and series."""
    h = 1e-4 * np.maximum(np.abs(p), 1e-2)
    rows = [p] + [p + np.eye(len(p))[i] * h[i] for i in range(len(p))]
    s = simulate_bands(np.array(rows)) / SIG[None, :, None]
    return ((s[:, :, 1:] - s[:, :, [0]]) / h).reshape(-1, len(p))


res_b = (OBS - fitted_b) / SIG
se_b, corr_b = standard_errors(jacobian_bands(fit_b), res_b.ravel(), 8)
compare = pd.DataFrame({
    "truth": np.r_[TRUTH, 1.4], "total only": np.r_[fit, np.nan],
    "total and bands": fit_b, "std error": se_b}, index=PB)
print(compare.to_string(float_format="{:,.4g}".format))
print("correlations with capacity_0:")
print(pd.Series(corr_b[2], index=PB).round(2).to_string())

Now the estimates are close to the truth: capacity of 4,152 patients a month (truth 3,957), activity at worst 0.353 (truth 0.35), growth rates of 2.8% and 1.8% a year (truth 3.0% and 2.0%). The standard error on capacity has fallen from about 125,000 to 118 patients a month. Referrals and capacity are still correlated at 1.00, because their gap is still what the total pins down, but the bands fix where the pair sits.

Not everything is recovered. `priority` comes out at 2.06 against 1.4, and the initiative at 8.2% against 10%. Our model is simpler than the process that made the data: it treats the three older bands alike, where the data used weights of 1.0, 0.9 and 0.8, and it ages patients continuously rather than once a month. The priority absorbs part of that difference. Estimates from a simplified model are estimates of *that model's* parameters, and can differ from the truth even with perfect data. This is **specification error**, and no amount of data removes it.

### A faster search for least squares

Profiling the band model with Nelder-Mead takes minutes, because every point needs a new search. When the payoff is a sum of squares, a faster method exists. **Gauss-Newton** uses the Jacobian to predict where the minimum is, as if the model were linear, and jumps there; the **Levenberg-Marquardt** version shortens the jump when a step fails. Because `run_batch` computes many runs at once, we can refit many data sets in a single loop: each step needs one batch for all the Jacobians and one to try the new points. We check it first: started from the first guess, does it reach the Nelder-Mead optimum?

In [ ]:
def refit_many(datasets, start, iters=12, hold=None):
    """Levenberg-Marquardt refits of many data sets at once.

    datasets: runs x months x 5. Each step needs one batch of runs for the
    Jacobians and one to try the new points, whatever the number of runs.
    start: one parameter vector, or one row per data set. hold: the index
    of a parameter to keep where it starts (for a profile).
    """
    B, k = len(datasets), np.shape(start)[-1]

    def resid(rows):
        sim = simulate_bands(rows).transpose(2, 0, 1)      # runs x months x 5
        return ((datasets - sim) / SIG).reshape(B, -1)

    p = np.array(np.broadcast_to(start, (B, k)), dtype=float)
    lam = np.full(B, 1e-3)
    r = resid(p)
    cost = (r ** 2).sum(axis=1)
    for _ in range(iters):
        h = 1e-4 * np.maximum(np.abs(p), 1e-2)
        shifted = np.concatenate([p + np.eye(k)[i] * h[:, [i]]
                                  for i in range(k)])
        sim = simulate_bands(shifted).transpose(2, 0, 1) / SIG
        sim = sim.reshape(k, B, -1)
        base = (datasets / SIG).reshape(B, -1) - r      # current fit / SIG
        S = ((sim - base[None]) / h.T[:, :, None]).transpose(1, 2, 0)
        if hold is not None:
            S[:, :, hold] = 0.0
        A = S.transpose(0, 2, 1) @ S                    # B x k x k
        g = (S.transpose(0, 2, 1) @ r[:, :, None])[:, :, 0]
        D = (A * np.eye(k) + 1e-12 * np.eye(k)) * lam[:, None, None]
        step = np.linalg.solve(A + D, g[:, :, None])[:, :, 0]
        trial = np.clip(p + step, LOW_B, HIGH_B)
        r_t = resid(trial)
        c_t = (r_t ** 2).sum(axis=1)
        better = c_t < cost
        p[better], r[better], cost[better] = trial[better], r_t[better], \
            c_t[better]
        lam = np.where(better, lam * 0.3, lam * 10)
    return p, cost


t0 = time.time()
check_p, check_cost = refit_many(OBS[None], np.r_[guess, 1.4])
print(f"Levenberg-Marquardt from the first guess: payoff {check_cost[0]:,.1f}"
      f" (Nelder-Mead {f_b:,.1f}) in {time.time() - t0:.1f} s")

It does: a payoff of 1,223.1 against Nelder-Mead's 1,223.0, in about a second. Now profile `capacity_0` for the band model, nine values in one call.

In [ ]:
cap_b = np.array([3700, 3800, 3900, 4000, 4100, 4200, 4300, 4400, 4500])
starts = np.tile(fit_b, (len(cap_b), 1))
starts[:, 0] += cap_b - fit_b[2]               # keep the gap: along the ridge
starts[:, 2] = cap_b
t0 = time.time()
_, prof_b = refit_many(np.repeat(OBS[None], len(cap_b), axis=0), starts,
                       iters=20, hold=2)
print(f"{time.time() - t0:.0f} s")
print(pd.Series(prof_b - f_b, index=cap_b,
                name="payoff above the best").round(1).to_string())
fig, ax = plt.subplots()
ax.plot(cap_values, prof_total - f_fit, marker="o", label="total only")
ax.plot(cap_b, prof_b - f_b, marker="s", label="total and bands")
ax.axvline(TRUTH[2], color="black", linestyle=":", label="truth")
ax.set(title="Profile of the payoff over capacity_0",
       xlabel="capacity_0 held fixed (patients/month)",
       ylabel="best payoff minus overall best")
ax.legend()
plt.show()

Figure: the payoff profile over `capacity_0` (patients per month) for the model calibrated to the total only (circles) and to the total and the bands (squares), with the truth marked by a dotted line. Alt text: one flat line along the bottom, and one U-shaped curve with its lowest point near 4,150.

The difference is stark. Calibrated to the total alone, the profile is flat. With the bands it is a clear valley: 37.4 above the best at 3,700 patients a month, 0.3 at 4,200 and 14.3 at 4,500. If the errors were independent, an approximate 95% interval would be where the curve stays within about 8 of its minimum (3.84 times the payoff per degree of freedom, about 2.07): roughly 4,000 to 4,400. That interval narrowly misses the truth, 3,957, for two reasons we have met: the errors are autocorrelated, so the interval is too narrow (section 11 fixes this), and the model is simplified.

**Try it 9.** *Check:* which estimates of the band model lie within two standard errors of the truth? *Apply:* calibrate to the four bands only, leaving out the total. Does `capacity_0` change? *Extend:* suppose a clinic audit says the priority is 1.4. Fix it there and refit. What happens to `capacity_0` and to the payoff, and why?

<details><summary>Worked answer</summary>

*Check:* `referrals_0`, `referral_growth`, `capacity_0`, `capacity_growth` and `activity_at_worst`; `recovery_time`, `initiative_boost` and `priority` do not. *Apply:* hardly: `capacity_0` is 4,154 and priority 2.06. The bands contain the total, so leaving it out changes only the weighting. *Extend:* `capacity_0` falls to 3,602, now further from the truth, and the payoff rises from 1,223.0 to 1,343.5. Fixing one parameter at its true value does not help when the model cannot represent the process exactly: the free priority was compensating for the older bands being treated alike. Outside evidence helps most when the model is structured to use it, for example with a separate weight for each band.
</details>

In [ ]:
inside = np.abs(fit_b - np.r_[TRUTH, 1.4]) <= 2 * se_b
print("check: within two standard errors of the truth:",
      [p for p, ok in zip(PB, inside) if ok])
f_bands_only = lambda p: payoff_bands(p, use=slice(1, None))
fit_bo, _, _ = nelder_mead(f_bands_only, fit_b, 0.02 * (HIGH_B - LOW_B))
print(f"apply: bands only: capacity_0 {fit_bo[2]:,.0f}, priority "
      f"{fit_bo[7]:.2f}")
k_pr = PB.index("priority")
f_audit = lambda z: payoff_bands(np.insert(z, k_pr, 1.4))
z, _, _ = nelder_mead(f_audit, np.delete(fit_b, k_pr),
                      np.delete(0.02 * (HIGH_B - LOW_B), k_pr))
fit_audit = np.insert(z, k_pr, 1.4)
print(f"extend: priority fixed at 1.4: capacity_0 {fit_audit[2]:,.0f}, "
      f"referrals_0 {fit_audit[0]:,.0f}, payoff "
      f"{payoff_bands(fit_audit):,.1f} (free: {f_b:,.1f})")

## 10. Out of sample: fit to 2014 to 2019, test 2020 onwards

A model that fits the data it was calibrated on has passed an easy test. The hard test is **out of sample**: calibrate on part of the data and predict the rest. For time series the split must be **in time**, so the model never sees the future it is asked to predict. A random split would let it interpolate.

We calibrate the one-stock model to 2014 to 2019 and test on 2020 to 2023. The model cannot learn about the shock from data that end before it, so we give it the shock and the initiative as they might be reported by the hospital: activity at 35% at worst, recovery over nine months, a 10% boost. Only referrals and capacity are fitted. And since section 8 suggests the capacity level may not be identifiable, we fit at four values of `capacity_0`.

In [ ]:
TRAIN, TEST = np.arange(72), np.arange(72, 120)     # 2014-2019, 2020-2023
REPORTED = {"activity_at_worst": 0.35, "recovery_time": 9,
            "initiative_boost": 0.10}
FREE = [0, 1, 3]                    # referrals_0, referral_growth, cap. growth


def train_fit(cap0):
    """Fit 2014-2019, capacity_0 held at cap0, the events as reported."""
    base = guess.copy()
    base[4:] = list(REPORTED.values())
    base[0], base[2] = cap0 + 50, cap0

    def f(z):
        p = base.copy()
        p[FREE] = z
        return payoff(p, months=TRAIN)
    z, fz, _ = nelder_mead(f, base[FREE], 0.05 * (HIGH - LOW)[FREE], 1500)
    p = base.copy()
    p[FREE] = z
    return p, fz


rows, paths = [], {}
for cap0 in [3600, 3950, 4300, 4650]:
    p, f_train = train_fit(cap0)
    sim = simulate(p)
    paths[cap0] = sim
    rows.append({"capacity_0 held at": cap0, "referrals_0": p[0],
                 "train payoff": f_train,
                 "train RMSE": np.sqrt(np.mean((sim - obs)[TRAIN] ** 2)),
                 "test RMSE": np.sqrt(np.mean((sim - obs)[TEST] ** 2))})
train_table = pd.DataFrame(rows)
print(train_table.to_string(index=False, float_format="{:,.1f}".format))
fig, ax = plt.subplots()
ax.plot(wl["month"], obs / 1000, color="black", label="data")
for cap0, sim in paths.items():
    ax.plot(wl["month"][TEST], sim[TEST] / 1000,
            label=f"fitted to 2014-2019, capacity_0 {cap0}")
ax.axvline(wl["month"][72], color="grey", linestyle=":")
ax.set(title="Four equally good fits, four different futures",
       xlabel="month", ylabel="patients waiting (thousands)")
ax.legend(fontsize=8)
plt.show()

Figure: the data (solid) and the 2020 to 2023 predictions of four models fitted to 2014 to 2019 with `capacity_0` held at 3,600, 3,950, 4,300 and 4,650 patients a month, in thousands. Alt text: four lines that start together in 2020 and fan out, some above and some below the data.

All four fit the training years identically: a payoff of 44.3 and an RMSE of 466 patients. Their test errors differ by a third, from 1,983 to 2,686 patients. Pre-2020 data cannot choose between them, and the reported size of the shock is applied to whichever capacity the fit happened to pick. **Identifiability limits prediction**, not only interpretation: a parameter the training data cannot see can still decide the forecast.

Now refit with more data. Calibrating the band model on 2014 to 2021, which includes the shock, and testing on 2022 to 2023 is a fairer test of the model as it will be used.

In [ ]:
TRAIN2, TEST2 = np.arange(96), np.arange(96, 120)   # 2014-2021, 2022-2023
fixed_boost = PB.index("initiative_boost")


def f_until_2021(z):
    return payoff_bands(np.insert(z, fixed_boost, 0.10), months=TRAIN2)


t0 = time.time()
z0 = np.delete(fit_b, fixed_boost)
step = np.delete(0.05 * (HIGH_B - LOW_B), fixed_boost)
z, _, _ = nelder_mead(f_until_2021, z0, step, 2000)
fit_2021 = np.insert(z, fixed_boost, 0.10)
sim_2021 = simulate_one(fit_2021)[:, 0]
print(f"{time.time() - t0:.0f} s")
for label, sim in [("fitted to 2014-2021", sim_2021),
                   ("fitted to all years", fitted_b[:, 0])]:
    m = measures(sim[TEST2], obs[TEST2])
    print(f"{label}: 2022-2023 RMSE {m['RMSE']:,.0f}, MAPE "
          f"{m['MAPE (%)']:.2f}%, U_M {m['U_M (bias)']:.2f}")
print("capacity_0 fitted to 2014-2021:", round(fit_2021[2]))

Fitted to 2014 to 2021, the model predicts 2022 and 2023 with an RMSE of 2,319 patients (3.20%), four times its in-sample error over the same months after the final refit on all years (564 patients). And 92% of the out-of-sample error is bias: the 2022 to 2023 level is systematically off, because the training data put capacity at 4,305 and the reported 10% boost was applied to that. Expect out-of-sample error to be several times in-sample error, and report it. The final model is the one refitted to all the data (section 9); the out-of-sample test tells us how far to trust its forecasts.

**Try it 10.** *Check:* express the four test RMSEs as a share of the average list in 2020 to 2023. *Apply:* repeat the fits at `capacity_0` of 3,600, 3,950 and 4,300 with training data that run to December 2020, so that they include the shock. Are the three still equally good? *Extend:* describe a **rolling-origin** test, in which the split moves forward a year at a time, and say what it would add.

<details><summary>Worked answer</summary>

*Check:* from 3.1% to 4.1% of the list. *Apply:* no. The payoffs are 54.2, 47.0 and 47.3: once the shock is in the training data, a capacity of 3,600 is clearly worse, though 3,950 and 4,300 still fit almost equally. Data that contain a disturbance identify more than data that do not. *Extend:* fit on 2014 to 2017 and test on 2018, then fit on 2014 to 2018 and test on 2019, and so on. It gives several test errors instead of one, shows whether the error grows with the forecast horizon, and shows whether the estimates drift as data accumulate, a sign of structure the model lacks.
</details>

In [ ]:
share = train_table["test RMSE"] / obs[TEST].mean()
print(f"check: test RMSE as a share of the mean list 2020-2023: "
      f"{share.min():.1%} to {share.max():.1%}")
TRAIN3 = np.arange(84)                          # 2014 to 2020
for cap0 in [3600, 3950, 4300]:
    base = fit.copy()
    base[4:] = list(REPORTED.values())
    base[0], base[2] = cap0 + 50, cap0
    f = lambda z: payoff(np.r_[z[0], z[1], cap0, z[2], base[4:]],
                         months=TRAIN3)
    z, fz, _ = nelder_mead(f, base[FREE], 0.05 * (HIGH - LOW)[FREE], 1500)
    print(f"apply: 2014-2020, capacity_0 {cap0}: payoff {fz:.1f}")

## 11. Uncertainty: a bootstrap of residual blocks

How uncertain are the estimates? The Jacobian's standard errors assume independent errors, which section 7 showed is false. The **bootstrap** [@efron1979] makes no such assumption. It builds many new data sets that could plausibly have been observed, refits the model to each, and reads the uncertainty from the spread of the refitted estimates. For a dynamic model the usual recipe resamples residuals [@dogan2007]:

1. Calibrate, and keep the fitted series and the residuals.
2. Make a new data set: the fitted series plus residuals drawn again from the old ones.
3. Refit, and record the estimates.
4. Repeat a few hundred times.

Drawing residuals one month at a time would destroy their autocorrelation and make the data look more informative than they are. A **block bootstrap** draws whole blocks of consecutive months (here 12) and strings them together, so the wandering pattern of the errors survives. The blocks are drawn for all five series at once, which keeps the links between bands. The batched Levenberg-Marquardt search refits all 200 data sets together.

In [ ]:
def block_bootstrap(residuals, block=12, rng=rng):
    """One resample of the rows of residuals, in blocks of whole months."""
    n = len(residuals)
    starts = rng.integers(0, n - block + 1, size=n // block + 1)
    rows = np.concatenate([np.arange(s, s + block) for s in starts])[:n]
    return residuals[rows]


B = 200
resid_b = OBS - fitted_b
datasets = np.stack([fitted_b + block_bootstrap(resid_b) for _ in range(B)])
t0 = time.time()
boot, boot_cost = refit_many(datasets, fit_b)
print(f"{B} refits in {time.time() - t0:.0f} s")
lo, hi = np.percentile(boot, [5, 95], axis=0)
print(pd.DataFrame({"truth": np.r_[TRUTH, 1.4], "estimate": fit_b,
                    "5%": lo, "95%": hi,
                    "naive +/-": 1.645 * se_b}, index=PB)
      .to_string(float_format="{:,.4g}".format))
print("bootstrap correlation, referrals_0 and capacity_0:",
      round(np.corrcoef(boot[:, 0], boot[:, 2])[0, 1], 3))

The 90% bootstrap intervals contain the truth for five of the eight parameters: both levels, both growth rates and activity at worst. Capacity, for example, runs from 3,853 to 4,392 patients a month, and the truth is 3,957. The intervals miss `recovery_time` (9.1 to 11.5 months, truth 9), `initiative_boost` (6.8% to 9.7%, truth 10%) and `priority` (1.70 to 2.32, truth 1.4). Those are the parameters the specification error pushes around, and no bootstrap can detect a model that is wrong in the same way in every replicate.

The last column shows the naive 90% interval from the Jacobian, plus or minus 1.645 standard errors. The bootstrap interval for capacity is about 40% wider. The naive one is too narrow because it treats the 120 correlated months as independent. In the bootstrap, `referrals_0` and `capacity_0` are still correlated at 0.998: their gap is known far better than either level.

## 12. From parameters to prediction bands

Uncertain parameters mean uncertain forecasts. A **Monte Carlo over the calibrated parameters** runs the model once for each bootstrap estimate, here to December 2025 with no change in policy. The spread of those runs is the uncertainty from the parameters alone. A **prediction band** for what will actually be observed must also include the noise the model does not explain, so we add a block of resampled residuals to each run.

In [ ]:
T_END = 143                                         # December 2025
future = pd.period_range("2014-01", periods=T_END + 1, freq="M")
future = future.to_timestamp()


def project(params, extra=0.0):
    rows = pd.DataFrame(params, columns=PB).assign(extra_capacity=extra)
    out = band_model.run_batch(rows, T=T_END, include=["Waiting list"],
                               warn_clips=False)
    return out.pivot(index="time", columns="run",
                     values="value").to_numpy()[::4]   # months x runs


runs = project(boot)
noise = np.stack([block_bootstrap(np.r_[resid_b[:, 0], resid_b[:24, 0]])
                  for _ in range(B)], axis=1)       # 144 months x runs
predicted = runs + noise
p_band = np.percentile(runs, [5, 50, 95], axis=1)
q_band = np.percentile(predicted, [5, 95], axis=1)
fig, ax = plt.subplots()
ax.fill_between(future, q_band[0] / 1000, q_band[1] / 1000, color="#56B4E9",
                alpha=0.35, label="90% prediction band")
ax.fill_between(future, p_band[0] / 1000, p_band[2] / 1000, color="#0072B2",
                alpha=0.6, label="90% band from parameters only")
ax.plot(future, p_band[1] / 1000, color="black", label="median")
ax.plot(wl["month"], obs / 1000, color="#D55E00", linestyle="--",
        label="data")
ax.set(title="The waiting list to December 2025, no change in policy",
       xlabel="month", ylabel="patients waiting (thousands)")
ax.legend(loc="upper left", fontsize=8)
plt.show()
print(f"Dec 2025: median {p_band[1, -1]:,.0f}; parameters only "
      f"{p_band[0, -1]:,.0f} to {p_band[2, -1]:,.0f}; with noise "
      f"{q_band[0, -1]:,.0f} to {q_band[1, -1]:,.0f} patients")

Figure: the waiting list to December 2025 with no change in policy, in thousands of patients: the median of 200 runs (solid), the 90% band from parameter uncertainty only (dark), the 90% prediction band including noise (light) and the data (dashed). Alt text: a narrow fan that starts at the end of the data in December 2023 and widens slightly to the right.

By December 2025 the median forecast is 74,599 patients, with a 90% band from the parameters alone of 73,302 to 76,000, and a prediction band of 73,365 to 76,436. The forecast of the *list* is tight even though the parameters are not: the data fix the gap between referrals and capacity well, and the list depends mainly on that gap.

The question a decision maker asks is different, though: what would more capacity achieve? That depends on the *level* of capacity, the very thing the total could not identify. Compare the effect of 5% more capacity from January 2024 under the band-calibrated model and under the equally good total-only fits along the ridge of section 8.

In [ ]:
effect = project(boot, extra=0.05)[-1] - runs[-1]
print("5% more capacity from January 2024, change in the list by Dec 2025:")
print("  calibrated to total and bands, 5th, 50th, 95th percentiles:",
      np.percentile(effect, [5, 50, 95]).round(-1))
ridge_rows = pd.DataFrame(ridge, columns=PARAMS)
on_ridge = []
for extra in (0.0, 0.05):
    out = one_stock.run_batch(ridge_rows.assign(extra_capacity=extra),
                              T=T_END, include=["Waiting list"])
    on_ridge.append(out.pivot(index="time", columns="run",
                              values="value").to_numpy()[-1])
print("  total only, along the ridge (capacity_0 3,600 to 4,800):",
      (on_ridge[1] - on_ridge[0]).round(-1))

With the bands, 5% more capacity shortens the list by 6,280 to 6,790 patients by December 2025 (median 6,480). With the total alone, seven fits that match the history equally well put the effect anywhere from 7,310 to 9,790 patients, a range wider than the whole band-based interval, and centred on a larger effect. The forecast of the status quo was tight in both cases; the forecast of the policy was not. **Calibrate for the question**: check identifiability for the parameters the decision depends on, not for the fit.

**Try it 12.** *Check:* how wide is the 90% bootstrap interval for `capacity_0`, compared with the naive one? *Apply:* redo the bootstrap with blocks of 1 month and of 24 months. What happens to the width for `capacity_0`? *Extend:* what does 10% more capacity from January 2024 achieve by December 2025?

<details><summary>Worked answer</summary>

*Check:* 539 patients a month, against 387 for the naive interval. *Apply:* 236 with blocks of 1 month, less than half, because single months break the autocorrelation and make the data look more informative than they are; 485 with blocks of 24 months, close to the 12-month result. Choose a block longer than the time the residuals take to forget themselves (about 5 months, section 7), and check that doubling it changes little. *Extend:* a reduction of 12,560 to 13,580 patients (median 12,960), about twice the effect of 5%. Over two years the response is close to linear in the size of the boost.
</details>

In [ ]:
lo_c, hi_c = np.percentile(boot[:, 2], [5, 95])
print(f"check: capacity_0, bootstrap 90% width {hi_c - lo_c:,.0f}; "
      f"naive {2 * 1.645 * se_b[2]:,.0f} patients/month")
for block in [1, 24]:
    data_k = np.stack([fitted_b + block_bootstrap(resid_b, block)
                       for _ in range(B)])
    boot_k, _ = refit_many(data_k, fit_b)
    lo_k, hi_k = np.percentile(boot_k[:, 2], [5, 95])
    print(f"apply: blocks of {block:2} months: capacity_0 90% width "
          f"{hi_k - lo_k:,.0f}")
effect10 = project(boot, extra=0.10)[-1] - runs[-1]
print("extend: 10% more capacity:", np.percentile(effect10, [5, 50, 95])
      .round(-1))

## Check it

Before you report a calibrated model:

* **Bounds.** Is any estimate on the edge of its plausible range? Then the data are pulling it somewhere implausible: find out why.
* **Several starts.** Do searches from different starting points agree on the estimates, not just the payoff?
* **Profiles.** Profile the payoff for every parameter the decision depends on. A flat profile is an unidentified parameter, however good the fit.
* **Measures.** Report RMSE or MAE in the data's units, the Theil fractions, and a benchmark. Never R² or MAPE alone.
* **Residuals.** Check their autocorrelation, and use a block bootstrap if they are correlated.
* **Out of sample.** Split in time, and expect the test error to be several times the training error.

## Running case

The waiting-list model is now calibrated. Fitted to the total alone, it matches the history to within 0.9% but cannot tell how much capacity the hospital has: any value from 3,600 to 4,800 patients a month fits. Adding the four bands of time waited fixes the level, at 4,152 patients a month in early 2014 (90% interval 3,853 to 4,392), and recovers the 2020 shock (activity at worst 0.35). The calibrated model forecasts 74,600 patients waiting by December 2025 with no change in policy (90% prediction band 73,400 to 76,400), and estimates that 5% more capacity from January 2024 would shorten the list by about 6,500 patients by then. Chapter 10 (From model to decision) uses this model to design and compare policies, and chapter 11 (Models that learn) sets it against machine-learning forecasts.

## Tell the decision maker

"We cannot see referrals or treatments directly, only the list. Using the list together with how long people have waited, we estimate that the hospital treated about 4,150 patients a month in early 2014, with capacity growing about 2% a year against referral growth of about 3%. On current trends the list will be about 74,600 by the end of 2025, most likely between 73,400 and 76,400. Adding 5% to capacity from January 2024 would cut that by about 6,500 patients, with a 90% range of 6,300 to 6,800. The data cannot measure how strongly urgent cases are prioritised; a short clinic audit would sharpen the estimates."

## Going further

* **Calibration as a test.** @oliva2003 argues that calibration is best seen as a set of structural tests, calibrating partial models where flows are known and checking whether the estimates are plausible, rather than as one big fit. Section 2's band bookkeeping is a small example.
* **Gradient matching.** When the noise is in the flows, as here, fitting the month-to-month changes rather than the levels gives errors closer to independent. It is a one-line change to the payoff and an instructive exercise.
* **Likelihood and Bayesian calibration.** Writing down a model of the errors turns the payoff into a likelihood; adding prior distributions for the parameters gives a Bayesian calibration, which handles bounds and outside evidence (such as a clinic audit) in one framework.
* **Bootstrap variants.** @dogan2007 compares bootstrap schemes for system dynamics models, including resampling the noise in the flows rather than the residuals in the stocks.
* **Specialist tools.** Larger models use gradient-based optimisers and dedicated calibration software; the logic of payoff, search, identifiability and uncertainty is the same as here.

## Mini-project

Improve the calibrated model and show that the improvement is real.

1. Give the treatment shares a separate weight for each of the four bands (three new parameters) and recalibrate to the total and the bands.
2. Add a seasonal swing to referrals. The equation language has no sine function, but a `TABLE` over `TIME` with one point per month, repeating the 12-month pattern, can carry a seasonal factor with its amplitude as a parameter.
3. For each change, report the payoff, the Theil fractions for every series, the residual autocorrelation and the out-of-sample error on 2022 to 2023.
4. Do the bootstrap intervals now contain the truth for every parameter? Write one paragraph for a decision maker on what changed and why it matters for the capacity question.

## Further reading

* @oliva2003, on calibration as a testing strategy.
* @barlas1996, on behaviour tests and their place in validation.
* @sterman2000, chapter 21, on summary statistics for historical fit, including the Theil decomposition.
* @dogan2007, on bootstrap confidence intervals for system dynamics parameters, and @efron1979, the bootstrap's origin.
* @nelder1965, the simplex method used in section 5.